# 02 — Évaluation de la qualité des données

Identifier les anomalies qui peuvent compromettre l'analyse et préparer les décisions de traitement.
Le notebook 01 décrit les sources ; ici, nous testons des règles explicites et examinons les lignes concernées.

**Kernel : Python (Telco Churn).** Les règles sont dans `configs/validation.yaml`,
les chemins et les cibles dans `configs/base.yaml` et les surcharges d'environnement.
Les données brutes restent intactes : aucune imputation, normalisation ou suppression.


## 1. Chargement et contrat de qualité

`ValidationConfig` valide les règles avec Pydantic ; `DataValidator` les applique.
Les noms des cibles sont repris de `Settings` : ils ne sont pas dupliqués dans le contrat.
Les colonnes supplémentaires sont acceptées, les colonnes attendues sont contrôlées.

Les catégories canoniques et les bornes métier sont des hypothèses documentées dans le YAML.
Un `warning` demande une investigation ; une `error` viole le contrat technique proposé.
Le notebook continue pour rendre toutes les anomalies visibles.


In [1]:
import pandas as pd
from IPython.display import display

from telco_churn.data.loaders import TelcoDataLoader
from telco_churn.data.schema import ValidationConfig
from telco_churn.data.validation import DataValidator
from telco_churn.logging import configure_logging
from telco_churn.settings import Settings

settings = Settings.load()
configure_logging(settings)
config = ValidationConfig.load(settings)
loader = TelcoDataLoader(settings)

datasets = {"churn": loader.load_raw("churn"), "pilot": loader.load_raw("pilot")}
targets = {"churn": settings.churn_target_column, "pilot": settings.pilot_target_column}
validators = {
    name: DataValidator(config.datasets[name], targets[name], config.example_limit)
    for name in datasets
}
results = {name: validators[name].assess(frame) for name, frame in datasets.items()}

print(f"Environnement : {settings.environment}")
print(f"Règles : {settings.validation_config_path.name}")
display(
    pd.DataFrame(
        {
            name: {
                "rows": len(frame),
                "columns": len(frame.columns),
                "target": targets[name],
                "unique_key": ", ".join(config.datasets[name].unique_key),
            }
            for name, frame in datasets.items()
        }
    ).T
)

2026-11-11 11:36:06 | INFO     | data.loaders | Dataset=churn path=/Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/data/raw/churn_clients.csv rows=38689 columns=16
2026-11-11 11:36:06 | INFO     | data.loaders | Dataset=pilot path=/Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/data/raw/pilote_campagne_2023.csv rows=4000 columns=10
2026-11-11 11:36:07 | INFO     | data.validation | Quality checks: rows=38689 rules=49 failed=12 skipped=0
2026-11-11 11:36:07 | INFO     | data.validation | Quality checks: rows=4000 rules=34 failed=0 skipped=0


Environnement : dev
Règles : validation.yaml


,rows,columns,target,unique_key
churn,38689,16,churn,"customer_id, snapshot_date"
pilot,4000,10,churn_30d,customer_id


### Lire les résultats

- `PASS` : aucune violation parmi les éléments évalués.
- `FAIL` : au moins une violation observée.
- `SKIP` : colonne absente ou aucune observation éligible ; ce n'est pas une réussite.

`n_checked` est le dénominateur du taux : lignes non manquantes pour les domaines,
lignes aux dates valides pour la chronologie, lignes à clé complète pour l'unicité.
Les règles de structure comptent les colonnes et la présence d'observations.
Les règles peuvent se recouvrir : **ne pas additionner les effectifs de lignes en échec**.


## 2. Structure, identifiants et doublons

Le principal est supposé avoir une ligne par `(customer_id, snapshot_date)` ; plusieurs
photographies du même client peuvent être légitimes. L'unicité de `customer_id` dans le
pilote reste une hypothèse à confirmer, signalée comme avertissement.

`duplicate_rows` compte les copies après la première occurrence.
`unique_key` compte toutes les lignes participant à une clé répétée ; les effectifs diffèrent donc.
Les clés incomplètes sont exclues de ce contrôle et signalées par `missing_values`.


In [2]:
overview_columns = [
    "rule",
    "column",
    "severity",
    "status",
    "n_checked",
    "n_failed",
    "failed_percent",
]
for name, checks in results.items():
    print(name.upper())
    mask = checks["rule"].isin(
        ["required_columns", "non_empty_dataset", "duplicate_rows", "unique_key"]
    )
    mask |= checks["rule"].eq("missing_values") & checks["column"].isin(
        [config.datasets[name].id_column, targets[name], *config.datasets[name].unique_key]
    )
    display(checks.loc[mask, overview_columns])

CHURN


,rule,column,severity,status,n_checked,n_failed,failed_percent
0,required_columns,schema,error,PASS,16,0,0.000000
1,non_empty_dataset,dataset,error,PASS,1,0,0.000000
2,duplicate_rows,"customer_id, snapshot_date, signup_date, age, ...",error,FAIL,38689,60,0.155083
3,missing_values,customer_id,error,PASS,38689,0,0.000000
4,missing_values,snapshot_date,error,PASS,38689,0,0.000000
18,missing_values,churn,error,PASS,38689,0,0.000000
19,unique_key,"customer_id, snapshot_date",error,FAIL,38689,120,0.310166


PILOT


,rule,column,severity,status,n_checked,n_failed,failed_percent
0,required_columns,schema,error,PASS,10,0,0.0
1,non_empty_dataset,dataset,error,PASS,1,0,0.0
2,duplicate_rows,"customer_id, age, contract, region, tenure_mon...",error,PASS,4000,0,0.0
3,missing_values,customer_id,error,PASS,4000,0,0.0
12,missing_values,churn_30d,error,PASS,4000,0,0.0
13,unique_key,customer_id,warning,PASS,4000,0,0.0


## 3. Valeurs manquantes : où se concentrent-elles ?

Le contrôle inclut aussi les chaînes vides ou composées d'espaces, au-delà des valeurs
manquantes reconnues par pandas. Les tableaux segmentés affichent les effectifs et les taux ;
une différence entre segments ne prouve pas le mécanisme à l'origine des manquants.
Les petites populations doivent être interprétées avec prudence. Les groupes restent bruts :
`mensuel` et `MENSUEL` ne sont pas fusionnés.


In [3]:
for name, checks in results.items():
    print(name.upper())
    display(
        checks.loc[
            checks["rule"].eq("missing_values") & checks["status"].ne("PASS"),
            overview_columns,
        ]
    )

missingness_tables = {}
for name, validator in validators.items():
    for group_column in config.datasets[name].group_columns:
        if group_column not in datasets[name]:
            print(f"{name} / {group_column} : regroupement impossible, colonne absente")
            continue
        table = validator.missingness_by_group(datasets[name], group_column)
        missingness_tables[(name, group_column)] = table
        if table["n_missing"].gt(0).any():
            print(f"{name.upper()} — par {group_column}")
            display(table.sort_values(["column", "missing_percent"], ascending=[True, False]))

CHURN


,rule,column,severity,status,n_checked,n_failed,failed_percent
6,missing_values,age,warning,FAIL,38689,2128,5.500271
11,missing_values,monthly_fee,warning,FAIL,38689,1897,4.903202
15,missing_values,nps,warning,FAIL,38689,26245,67.835819


PILOT


,rule,column,severity,status,n_checked,n_failed,failed_percent


CHURN — par snapshot_date


,group,column,n_rows,n_missing,missing_percent
30,2024-11-30,age,3006,181,6.021291
27,2024-10-31,age,3092,184,5.950841
33,2024-12-31,age,2915,170,5.831904
18,2024-07-31,age,3223,184,5.708967
15,2024-06-30,age,3255,185,5.683564
0,2024-01-31,age,3494,193,5.523755
9,2024-04-30,age,3305,179,5.416036
24,2024-09-30,age,3134,168,5.360562
6,2024-03-31,age,3363,177,5.263158
3,2024-02-29,age,3426,179,5.224752


CHURN — par contract


,group,column,n_rows,n_missing,missing_percent
33,mensuel,age,209,19,9.090909
12,MENSUEL,age,197,15,7.614213
15,Mensuel,age,225,16,7.111111
27,biannuel,age,59,4,6.779661
24,biannuel,age,5966,340,5.698961
18,annuel,age,12175,675,5.544148
3,Annuel,age,112,6,5.357143
30,mensuel,age,19359,1037,5.356682
0,ANNUEL,age,129,6,4.651163
21,annuel,age,119,5,4.201681


CHURN — par region


,group,column,n_rows,n_missing,missing_percent
6,GES,age,5365,320,5.964585
3,BRE,age,5455,318,5.829514
0,ARA,age,5807,331,5.700017
18,PACA,age,5315,297,5.587959
15,OCC,age,5498,302,5.492907
9,IDF,age,5598,280,5.001786
12,NAQ,age,5651,280,4.954875
13,NAQ,monthly_fee,5651,300,5.308795
4,BRE,monthly_fee,5455,281,5.151237
10,IDF,monthly_fee,5598,281,5.019650


## 4. Types et cohérence métier

Les nombres doivent être interprétables, finis et dans le domaine déclaré. Les compteurs,
les indicateurs binaires et les cibles doivent être entiers. Les nombres stockés en chaînes
sont signalés séparément, sans convertir la source.

Les catégories sont comparées exactement aux modalités proposées dans le YAML.
Le contrôle de format repère aussi les écarts de casse et d'espaces. La borne d'âge 18–120,
les montants non négatifs et l'interprétation de `nps` comme note individuelle 0–10 sont
à confirmer : ils génèrent des avertissements. Aucun plafond arbitraire n'est imposé à `data_gb`.


In [4]:
for name, checks in results.items():
    print(name.upper())
    mask = checks["rule"].isin(["numeric_type", "numeric_domain", "category", "category_format"])
    issues = checks.loc[mask & checks["status"].ne("PASS")]
    if issues.empty:
        print("Aucune anomalie détectée par les règles de domaine configurées.")
    else:
        display(issues[overview_columns + ["examples", "action"]])

CHURN


,rule,column,severity,status,n_checked,n_failed,failed_percent,examples,action
23,numeric_domain,age,warning,FAIL,36561,364,0.995596,"[{'row_position': 22, 'age': -1.0}, {'row_posi...",Hypothèse de plausibilité à confirmer ; aucun ...
25,numeric_domain,monthly_fee,warning,FAIL,36792,113,0.307132,"[{'row_position': 234, 'monthly_fee': -62.72},...",Montant négatif à expliquer : erreur ou ajuste...
40,category,contract,warning,FAIL,38689,1189,3.073225,"[{'row_position': 66, 'contract': 'MENSUEL'}, ...","Modalités canoniques proposées : ['mensuel', '..."
41,category_format,contract,warning,FAIL,38689,1189,3.073225,"[{'row_position': 66, 'contract': 'MENSUEL'}, ...","Vérifier les différences de casse/espaces, san..."
42,category,channel,warning,FAIL,38689,1139,2.943989,"[{'row_position': 27, 'channel': 'BOUTIQUE'}, ...","Modalités canoniques proposées : ['web', 'bout..."
43,category_format,channel,warning,FAIL,38689,1139,2.943989,"[{'row_position': 27, 'channel': 'BOUTIQUE'}, ...","Vérifier les différences de casse/espaces, san..."


PILOT
Aucune anomalie détectée par les règles de domaine configurées.


## 5. Cohérence et couverture temporelles

Vérifier la lecture des dates et `signup_date ≤ snapshot_date` sur les paires valides.
Les valeurs non interprétables sont rapportées séparément ; elles ne sont pas transformées dans la source.
Les bornes ci-dessous décrivent la couverture observée, sans présumer la fréquence attendue.
Le pilote ne contient pas de date configurée : son nom de fichier ne suffit pas à valider sa période.


In [5]:
for name, checks in results.items():
    print(name.upper())
    if not config.datasets[name].dates:
        print("Pas de date disponible pour vérifier la couverture temporelle.")
        continue
    display(validators[name].temporal_coverage(datasets[name]))
    display(
        checks.loc[
            checks["rule"].isin(["date_parse", "date_order"]), overview_columns + ["examples"]
        ]
    )

CHURN


,column,first_date,last_date,n_valid,n_distinct
0,snapshot_date,2024-01-31,2024-12-31,38689,12
1,signup_date,2019-01-02,2026-02-04,38689,1787


,rule,column,severity,status,n_checked,n_failed,failed_percent,examples
46,date_parse,snapshot_date,error,PASS,38689,0,0.000000,[]
47,date_parse,signup_date,error,PASS,38689,0,0.000000,[]
48,date_order,"signup_date, snapshot_date",error,FAIL,38689,173,0.447156,"[{'row_position': 480, 'signup_date': '2025-03..."


PILOT
Pas de date disponible pour vérifier la couverture temporelle.


## 6. Synthèse des anomalies et actions proposées

Les exemples contiennent la **position de ligne en base zéro**, utilisable avec `df.iloc`,
et quelques valeurs brutes. Ils ne représentent pas toutes les anomalies.
Les erreurs et avertissements sont des constats, pas une décision automatique de nettoyage.


In [6]:
all_checks = pd.concat(
    [table.assign(dataset=name) for name, table in results.items()], ignore_index=True
)
display(all_checks.groupby(["dataset", "severity", "status"]).size().rename("n_checks").to_frame())

anomalies = all_checks.loc[all_checks["status"].eq("FAIL")].copy()
display(anomalies[["dataset", *overview_columns, "examples", "action"]])

skipped = all_checks.loc[all_checks["status"].eq("SKIP")]
if not skipped.empty:
    print("Contrôles non évalués — ne pas les considérer comme validés")
    display(skipped[["dataset", "rule", "column", "action"]])

n_checks
dataset severity status          
churn   error    FAIL           3
                 PASS          20
        warning  FAIL           9
                 PASS          17
pilot   error    PASS          17
        warning  PASS          17

,dataset,rule,column,severity,status,n_checked,n_failed,failed_percent,examples,action
2,churn,duplicate_rows,"customer_id, snapshot_date, signup_date, age, ...",error,FAIL,38689,60,0.155083,"[{'row_position': 38629, 'customer_id': 101357...",Examiner les copies après la première occurren...
6,churn,missing_values,age,warning,FAIL,38689,2128,5.500271,"[{'row_position': 4, 'age': nan}, {'row_positi...",Identifier l'origine des manquants ; aucune im...
11,churn,missing_values,monthly_fee,warning,FAIL,38689,1897,4.903202,"[{'row_position': 7, 'monthly_fee': nan}, {'ro...",Identifier l'origine des manquants ; aucune im...
15,churn,missing_values,nps,warning,FAIL,38689,26245,67.835819,"[{'row_position': 1, 'nps': nan}, {'row_positi...",Identifier l'origine des manquants ; aucune im...
19,churn,unique_key,"customer_id, snapshot_date",error,FAIL,38689,120,0.310166,"[{'row_position': 612, 'customer_id': 100697, ...",Hypothèse : une observation par client et par ...
23,churn,numeric_domain,age,warning,FAIL,36561,364,0.995596,"[{'row_position': 22, 'age': -1.0}, {'row_posi...",Hypothèse de plausibilité à confirmer ; aucun ...
25,churn,numeric_domain,monthly_fee,warning,FAIL,36792,113,0.307132,"[{'row_position': 234, 'monthly_fee': -62.72},...",Montant négatif à expliquer : erreur ou ajuste...
40,churn,category,contract,warning,FAIL,38689,1189,3.073225,"[{'row_position': 66, 'contract': 'MENSUEL'}, ...","Modalités canoniques proposées : ['mensuel', '..."
41,churn,category_format,contract,warning,FAIL,38689,1189,3.073225,"[{'row_position': 66, 'contract': 'MENSUEL'}, ...","Vérifier les différences de casse/espaces, san..."
42,churn,category,channel,warning,FAIL,38689,1139,2.943989,"[{'row_position': 27, 'channel': 'BOUTIQUE'}, ...","Modalités canoniques proposées : ['web', 'bout..."


### Export optionnel

Passer `export_reports` à `True` pour enregistrer la synthèse complète dans
`artifacts/metrics/data_quality_checks.csv`. Les exemples de lignes restent dans le notebook.


In [7]:
export_reports = False
if export_reports:
    output_path = settings.project_root / "artifacts/metrics/data_quality_checks.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    all_checks.drop(columns="examples").to_csv(output_path, index=False)
    print(f"Rapport enregistré : {output_path}")

## 7. Décisions à documenter avant le preprocessing

À compléter à partir des résultats, après validation métier :

- Confirmer l'unité d'observation et le traitement des doublons exacts et des clés répétées.
- Expliquer les âges improbables et les frais mensuels négatifs avant de proposer une correction.
- Confirmer les modalités canoniques et les règles de normalisation des catégories.
- Définir le traitement des manquants, notamment de `nps`, sans imputation globale prématurée.
- Confirmer les définitions des cibles, leurs horizons et leur disponibilité temporelle.

Le succès d'un contrôle signifie uniquement que les données respectent la règle configurée.
Ce notebook ne démontre ni l'absence de fuite de cible ni la validité causale du pilote.
Les stratégies de correction et de préparation seront développées dans les étapes suivantes.
